In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", None)

column_types = {
    "id": "Int64",
    "osm_id": "str",
    "addr_street": "str",
    "addr_number": "str",
    "ground_floor_group": "str",
    "ground_floor_function": "str",
    "upper_floor_group": "str",
    "upper_floor_function": "str",
    "number_of_storeys": "Int64",
    "is_mixed_use": "boolean",
    "build_year_est": "Int64",
    "arch_style": "str",
    "condition": "Int64",
    "max_height": "float64",
    "is_heritage": "str",
    "source": "str",
    "confidence": "Int64",
    "date_of_checking": "str",
}

In [2]:
data_gdynia = pd.read_csv(
    "../../data/processed/fixed/srodmiescie.csv",
    sep=";",
    index_col=0,
    dtype=column_types,
    parse_dates=["date_of_checking"],
)
data_gdynia.head()

,osm_id,addr_street,addr_number,ground_floor_group,ground_floor_function,upper_floor_group,upper_floor_function,number_of_storeys,is_mixed_use,build_year_est,arch_style,condition,max_height,is_heritage,source,confidence,date_of_checking,number_of_functions
id,,,,,,,,,,,,,,,,,,
1,127184291,Świętego Piotra,8,NaN,hotel,NaN,Hotel,7,<NA>,2000,współczesność,3,NaN,NaN,google.com/maps,3,2026-09-27,1
2,890729731,Świętego Piotra,6D,NaN,mieszkaniowa wielorodzinna,NaN,Mieszkaniowa Wielorodzinna,5,<NA>,2020,współczesność,3,NaN,NaN,google.com/maps,3,2026-09-27,1
3,890292040,Węglowa,22,NaN,centrum handlowe,NaN,Mieszkaniowa Wielorodzinna,17,<NA>,2020,współczesność,3,NaN,NaN,google.com/maps,3,2026-09-27,1
4,890729732,Świętego Piotra,6,NaN,mieszkaniowa wielorodzinna,NaN,Mieszkaniowa Wielorodzinna,17,<NA>,2020,współczesność,3,NaN,NaN,google.com/maps,3,2026-09-27,1
5,136885205,Tadeusza Wendy,15,NaN,centrum medyczne,NaN,Centrum Medyczne,6,<NA>,2000,współczesność,3,NaN,NaN,google.com/maps,2,2026-09-27,1


In [3]:
data_gdynia_long = pd.read_csv(
    "../../data/processed/fixed/srodmiescie_long.csv",
    sep=";",
    index_col=0,
    dtype=column_types,
)
data_gdynia_long.head()

,addr_street,ground_floor_function
id,,
1,Świętego Piotra,hotel
2,Świętego Piotra,mieszkaniowa wielorodzinna
3,Węglowa,centrum handlowe
4,Świętego Piotra,mieszkaniowa wielorodzinna
5,Tadeusza Wendy,centrum medyczne


In [4]:
category_map = pd.read_csv(
    "../../data/reference/category_map.csv",
    sep=";",
    index_col=0,
    dtype=column_types,
)

In [5]:
data_long_categories = pd.merge(
    data_gdynia_long,
    category_map,
    left_on="ground_floor_function",
    right_index=True,
    indicator=True,
    how="outer",
    validate="many_to_one",
)

In [6]:
assert len(data_long_categories.loc[(data_long_categories["_merge"] != "both")]) == 0
assert data_long_categories["category"].notna().all()

In [7]:
data_long_categories = data_long_categories.loc[
    :, ["addr_street", "function_clean", "category"]
]
data_long_categories

,addr_street,function_clean,category
id,,,
219,Stefana Żeromskiego,administracja,organizacje publiczne
258,3 Maja,administracja,organizacje publiczne
374,Starowiejska,administracja,organizacje publiczne
495,10 Lutego,administracja,organizacje publiczne
86,Jana z Kolna,administracja,organizacje publiczne
...,...,...,...
1209,Władysława IV,sklep spożywczy,handel
1222,Świętojańska,sklep spożywczy,handel
1234,Świętojańska,sklep spożywczy,handel


In [8]:
data_long_categories = (
    data_long_categories.groupby(["addr_street", "category"], as_index=False)
    .agg("size")
    .rename(columns={"size": "num_premise"})
    .assign(
        street_total=lambda d: d.groupby("addr_street")["num_premise"].transform("sum")
    )
    .assign(share=lambda d: d["num_premise"] / d["street_total"])
)
data_long_categories

,addr_street,category,num_premise,street_total,share
0,10 Lutego,brak parteru,1,64,0.015625
1,10 Lutego,finanse,4,64,0.062500
2,10 Lutego,gastronomia,8,64,0.125000
3,10 Lutego,handel,30,64,0.468750
4,10 Lutego,mieszkaniowa,1,64,0.015625
...,...,...,...,...,...
261,Świętojańska,zdrowie,9,253,0.035573
262,Żołnierzy I Armii Wojska Polskiego,gastronomia,1,4,0.250000
263,Żołnierzy I Armii Wojska Polskiego,handel,1,4,0.250000
264,Żołnierzy I Armii Wojska Polskiego,usługi,1,4,0.250000


In [9]:
sum_shares_street = data_long_categories.groupby("addr_street").agg(
    sum_1=("share", "sum")
)


In [10]:
assert np.isclose(sum_shares_street["sum_1"], 1).all()